# FELIX hit sweep — every phenotype, every arm

Walks the whole run table, screens every genome-wide hit in all three arms, and puts the surviving candidates in one place.

**Why a sweep and not 64 single-phenotype reports.** Two questions only answerable across phenotypes:

- *Does a locus recur?* A site that surfaces as a hit in several unrelated phenotypes is usually a property of the locus — mapping, genotyping, a degenerate dosage distribution — rather than real pleiotropy. Recurrence is a screen in its own right.
- *How does the artifact rate scale?* The SPA fallback bites hardest at low case counts. Plotting hit count against case count over the full set shows where the repeat arm stops being usable, which is what a case floor has to be set from.

**Cost.** Hit tables for the repeat arms exist only inside `run_summary_tar` (the table's `top_hits_tables` column points at the main arm alone). The sweep streams each bundle through `tar` and keeps only `calibration_summary.tsv` and `top_hits.tsv` — about 0.2 MB per run instead of 125 MB, and the tarball never lands on disk. Harvesting is cached per run, so a second pass is free.

**Positions, not marker IDs.** The SNV arm writes `chr9_8991058_T_TA;…` where the repeat arms write `9-8991058-8991069-A` for the same site, so cross-arm and cross-phenotype matching is done by clustering on position.

## 1 · Setup

In [ ]:
import os, json, subprocess, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

ETYPE    = "flare_lai_prod_set"
ID_COL   = f"entity:{ETYPE}_id"
TAR_COL  = "run_summary_tar"
NUM_ANCS = 5
STORE    = "harvest_cache"      # small per-run extracts; delete to re-harvest

WS_NS   = os.environ.get("WORKSPACE_NAMESPACE")
WS_NAME = os.environ.get("WORKSPACE_NAME")
BUCKET  = os.environ.get("WORKSPACE_BUCKET")
print("workspace:", WS_NS, "/", WS_NAME)
assert WS_NS and WS_NAME, "set WS_NS / WS_NAME by hand if the env vars are absent"

# One canonical copy of the analysis module: felix/scripts/, rsynced to the
# bucket by 00_sync_repo.
#
# ALWAYS re-copy. felix_07 writes this same filename into the working
# directory via %%writefile, and an older sync leaves an older copy, so a
# "fetch only if absent" guard silently imports stale code.
MODULE = Path("felix_phenotype_report.py")
SRC = f"{BUCKET}/felix/scripts/felix_phenotype_report.py"
subprocess.run(["gsutil", "cp", SRC, str(MODULE)], check=True)

import importlib
importlib.invalidate_caches()
sys.modules.pop("felix_phenotype_report", None)   # drop anything already imported
import felix_phenotype_report as fpr

REQUIRED = ["sweep", "harvest", "cluster_positions", "recurrence", "corroborated",
            "hit_report", "calibration_report", "spa_fallback", "excess_het", "lambda_gc"]
missing = [f for f in REQUIRED if not hasattr(fpr, f)]
if missing:
    raise RuntimeError(
        f"{MODULE} is missing {missing}.\n"
        f"The copy at {SRC} predates the functions listed (the sweep "
        f"functions arrived in a67ebf5, the excess-heterozygosity screen "
        f"after 180394d). Re-run 00_sync_repo (it rsyncs felix/scripts/ to "
        f"the bucket from origin/main), then re-run this cell.")
print(f"module ok ({MODULE.stat().st_size // 1024} KB):", ", ".join(REQUIRED))

## 2 · Run table

In [ ]:
from firecloud import api as fapi

def entities_to_df(ns, name, etype):
    r = fapi.get_entities(ns, name, etype); r.raise_for_status()
    rows = []
    for e in r.json():
        rec = {f"entity:{etype}_id": e["name"]}
        for k, v in e.get("attributes", {}).items():
            if isinstance(v, dict) and "items" in v:
                v = list(v["items"])
            rec[k] = v
        rows.append(rec)
    return pd.DataFrame(rows)

runs = entities_to_df(WS_NS, WS_NAME, ETYPE).sort_values(ID_COL).reset_index(drop=True)
done = runs[runs[TAR_COL].notna()].copy()
print(f"{len(runs)} rows in {ETYPE}; {len(done)} have {TAR_COL}")

## 3 · Sweep every run

In [ ]:
%%time
cal, hits, failures = fpr.sweep(done, ID_COL, TAR_COL, store=STORE,
                                num_ancs=NUM_ANCS, workers=8)
print(f"\n{len(cal)} run x arm calibration rows; {len(hits)} genome-wide hits; "
      f"{len(failures)} harvest failures")
if len(failures):
    display(failures)
size_mb = sum(f.stat().st_size for f in Path(STORE).rglob("*") if f.is_file()) / 1e6
print(f"harvest cache: {size_mb:.1f} MB")

## 4 · Per-phenotype summary

In [ ]:
# Per-phenotype summary: one row per run, arms side by side
wide = cal.pivot_table(index=["run", "n_cases", "N_eff"], columns="arm",
                       values=["lambda_gc", "n_gw_sig"], aggfunc="first").reset_index()
wide.columns = [c[0] if not c[1] else f"{c[0]}_{c[1]}" for c in wide.columns]

if len(hits):
    kept = (hits[hits.verdict == "keep"].groupby(["run", "arm"]).size()
            .unstack(fill_value=0).add_prefix("kept_").reset_index())
    wide = wide.merge(kept, on="run", how="left")
for c in [c for c in wide.columns if c.startswith("kept_")]:
    wide[c] = wide[c].fillna(0).astype(int)

wide = wide.sort_values("n_cases")
display(Markdown(f"### {len(wide)} runs, sorted by case count"))
display(wide)

tot_rep  = int(cal.n_gw_sig.sum())
tot_keep = int((hits.verdict == "keep").sum()) if len(hits) else 0
tot_excl = int((hits.verdict == "EXCLUDE").sum()) if len(hits) else 0
display(Markdown(
    f"**{tot_rep} hits reported across all runs and arms → {tot_excl} excluded, "
    f"{tot_keep} retained.** "
    f"({tot_rep - tot_excl - tot_keep} reported hits fall outside the bundles' "
    f"top-50 files and could not be screened.)"))

# Exclusions are not all SPA fallbacks — break them out.
if len(hits):
    by_reason = (hits.loc[hits.verdict == "EXCLUDE", "reasons"]
                 .str.split(",").explode().value_counts().rename("n_hits").to_frame())
    display(Markdown("Exclusions by reason (one hit can trigger more than one):"))
    display(by_reason)

In [ ]:
# Genotype check: did the excess-heterozygosity screen see every hit it can?
# main and ru_biallelic hits carry ctrl_F = 1 - (observed / Hardy-Weinberg heterozygotes) among controls; the dosage arm has none by design.
if len(hits) and "ctrl_F" in hits.columns:
    chk = hits.assign(screened=hits.ctrl_F.notna())
    display(Markdown("Hits with a control genotype check, by arm:"))
    display(chk.groupby("arm").screened.agg(n_hits="size", with_ctrl_F="sum"))
    missing = chk[(chk.arm != "ru") & ~chk.screened]
    if len(missing):
        display(Markdown(f"**{len(missing)} main or ru_biallelic hits have no control counts in top_hits, so the screen did not run on them.**"))
    display(Markdown("Excluded for excess heterozygosity, by arm:"))
    display(chk[chk.reasons.str.contains("excess_het")].groupby("arm").size().rename("n_hits").to_frame())
    fig, ax = plt.subplots(figsize=(6.5, 3))
    ax.hist(chk.loc[chk.screened, "ctrl_F"].clip(-1.05, 1.05), bins=np.linspace(-1.05, 1.05, 43), color="#3b6ea5")
    ax.axvline(fpr.EXCESS_HET_F, ls="--", lw=0.8, color="#d1495b")
    ax.set_xlabel("controls' F = 1 - observed / Hardy-Weinberg heterozygotes (dashed: exclusion cut)"); ax.set_ylabel("hits")
    fig.tight_layout(); plt.show()
    kept = chk[(chk.verdict == "keep") & chk.screened]
    display(Markdown("ctrl_F over the retained hits (the cut should sit in a gap, not inside this distribution):"))
    display(kept.ctrl_F.describe()[["count", "min", "25%", "50%", "75%", "max"]].to_frame().T)
else:
    display(Markdown("`hits` has no ctrl_F column: the module copy is stale. Re-run 00_sync_repo, then the setup cell."))


## 5 · Cross-phenotype views

In [ ]:
display(Markdown("### Loci hit in more than one phenotype"))
rec = fpr.recurrence(hits, min_phenotypes=2)
if len(rec):
    display(rec.head(40))
    display(Markdown("> A locus recurring across unrelated phenotypes is more often a "
                     "property of the locus than real pleiotropy. Check these before "
                     "treating any of them as findings."))
else:
    display(Markdown("None — no locus is a hit in two or more phenotypes."))

display(Markdown("### Retained hits corroborated across arms (same phenotype)"))
cor = fpr.corroborated(hits)
if len(cor):
    display(cor)
    display(Markdown("> Independent codings of the same data agreeing is the strongest "
                     "evidence available short of replication."))
else:
    display(Markdown("None."))

display(Markdown("### Every retained hit"))
if len(hits) and (hits.verdict == "keep").any():
    display(hits[hits.verdict == "keep"].sort_values("p")
            [["run", "phenotype", "arm", "CHR", "POS", "MarkerID", "p", "case_carriers"]])
else:
    display(Markdown("None retained."))

## 6 · Scaling with case count

In [ ]:
ru = cal[cal.arm == "ru"].copy()
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))

ax[0].scatter(ru.n_cases, ru.lambda_gc, s=20, color="#3b6ea5")
ax[0].axhline(1.0, ls="--", lw=0.8, color="#888888")
ax[0].set_xscale("log"); ax[0].set_xlabel("cases")
ax[0].set_ylabel(r"$\lambda$"); ax[0].set_title("Calibration vs case count (ru arm)")

ax[1].scatter(ru.n_cases, ru.n_gw_sig + 0.5, s=20, color="#d1495b")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("cases"); ax[1].set_ylabel("reported hits + 0.5")
ax[1].set_title("Hits before screening (ru arm)")

if len(hits):
    per = (hits.groupby(["run", "arm"]).verdict
             .value_counts().unstack(fill_value=0).reset_index())
    per = per[per.arm == "ru"].merge(ru[["run", "n_cases"]], on="run", how="left")
    if "EXCLUDE" in per.columns:
        ax[2].scatter(per.n_cases, per["EXCLUDE"] + 0.5, s=20,
                      color="#d1495b", label="flagged")
    if "keep" in per.columns:
        ax[2].scatter(per.n_cases, per["keep"] + 0.5, s=28, facecolors="none",
                      edgecolors="#2e7d5b", label="retained")
    ax[2].set_xscale("log"); ax[2].set_yscale("log")
    ax[2].set_xlabel("cases"); ax[2].set_ylabel("hits + 0.5")
    ax[2].set_title("After screening (ru arm)"); ax[2].legend(frameon=False)
fig.tight_layout(); plt.show()

## 7 · Export

In [ ]:
cal.to_csv("felix_sweep_calibration.tsv", sep="\t", index=False)
wide.to_csv("felix_sweep_by_phenotype.tsv", sep="\t", index=False)
if len(hits):
    hits.to_csv("felix_sweep_all_hits.tsv", sep="\t", index=False)
    fpr.recurrence(hits).to_csv("felix_sweep_recurrent_loci.tsv", sep="\t", index=False)
    fpr.corroborated(hits).to_csv("felix_sweep_corroborated.tsv", sep="\t", index=False)
print("wrote:", sorted(p.name for p in Path(".").glob("felix_sweep_*.tsv")))
if BUCKET:
    print(f"copy up with:  gsutil -m cp felix_sweep_*.tsv {BUCKET}/analysis/")

## 8 · Figure 3 inputs

Two extras for the manuscript figure, run after the export cell: the region extracts for the three example loci (needs `done`, `ID_COL`, `TAR_COL` from above, and `felix/scripts/extract_region.py` in the bucket via `00_sync_repo`), and the best screened P of every run for the PheWAS panel (needs `sweep_best_p.py` the same way).

In [ ]:
# Region extracts for the Figure 3 candidate review: one 1 Mb window per locus out of the run's three merged result files
# (main, ru, ru_biallelic), streamed with gsutil cat, so a few MB come back instead of the 0.5-27 GB tables.
# LOCI: the candidate loci of manuscript_figures/fig3_candidates (an SV, a long insertion or deletion, or a repeat expansion) that have no extract
# yet: the retained ones, then the pending ones (excluded only by the earlier conditioned spa_fallback rule) that have no region flag or show the
# repeat-unit dosage test ahead of the allele dosage test. Regenerate with fig3_candidates.terra_windows() when the pool changes. Already in the
# manuscript data/: BI_174, BI_177, BI_177_2, CA_103_22, CV_413, CV_431_1, CV_438, CV_440, CV_446_2, EM_232_2, GE_983_2.
import subprocess, os
BUCKET = os.environ["WORKSPACE_BUCKET"]
LOCI = [  # (run, chromosome, start, end): 1 Mb windows, hg38
    ("felix.prod.BI_168_1", "17", 36_740_000, 37_740_000),
    ("felix.prod.BI_164_2", "21", 23_420_000, 24_420_000),
    ("felix.prod.CV_413_11", "1", 57_620_000, 58_620_000),
    ("felix.prod.GI_520_11", "12", 79_100_000, 80_100_000),
    ("felix.prod.DE_679", "11", 41_190_000, 42_190_000),
    ("felix.prod.GU_610_1", "1", 53_690_000, 54_690_000),
    ("felix.prod.ID_052", "11", 82_670_000, 83_670_000),
    ("felix.prod.GU_582", "16", 1_540_000, 2_540_000),
    ("felix.prod.CV_425", "17", 74_880_000, 75_880_000),
    ("felix.prod.GU_608", "2", 233_620_000, 234_620_000),
]
subprocess.run(["gsutil", "cp", f"{BUCKET}/felix/scripts/extract_region.py", "extract_region.py"], check=True)   # always re-copy
for RUN, CHROM, START, END in LOCI:
    row = done[done[ID_COL] == RUN]
    assert len(row) == 1, [r for r in done[ID_COL] if RUN.split(".")[-1] in r]
    root = row[TAR_COL].iloc[0].split("/call-Bundle")[0]
    tag = "fig3_locus_" + RUN.removeprefix("felix.prod.")
    print(RUN, "workflow root:", root)
    rc = subprocess.run(["python3", "extract_region.py", "--root", root, "--chrom", CHROM, "--start", str(START), "--end", str(END),
                         "--out-prefix", tag]).returncode
    if rc:  # an arm without a merged table: say so and carry on with the next locus
        print(f"  -> extract_region.py exited {rc}; see the lines above. Continuing.")
subprocess.run("ls -la fig3_locus_*.tsv && wc -l fig3_locus_*.tsv", shell=True)

In [ ]:
# Best screened P of EVERY run and arm, for the Figure 3 PheWAS panel. The sweep above keeps only hits (P < 5e-8), so a phenotype
# with no hit has no P in any table; this reads the same harvest cache (STORE), with no new download.
import subprocess, sys
subprocess.run(["gsutil", "cp", f"{BUCKET}/felix/scripts/sweep_best_p.py", "sweep_best_p.py"], check=True)   # always re-copy
sys.modules.pop("sweep_best_p", None)
import sweep_best_p as sbp

best = sbp.sweep_best_p(STORE, num_ancs=NUM_ANCS)
print(f"{len(best)} run x arm rows; {best.run.nunique()} runs; {int(best.censored.sum())} arms with no screenable test (censored)")
print(best.groupby("arm").agg(runs=("run", "nunique"), censored=("censored", "sum"), median_rows=("n_rows", "median")))
best.to_csv("felix_sweep_best_p.tsv", sep="\t", index=False)

## 9 · Bundle for download

One tarball of everything the figure reads: the sweep tables, `felix_sweep_best_p.tsv`, and the nine `fig3_locus_*` region files.

In [ ]:
!rm -f felix_sweep.tar.gz felix_sweep.tar
!tar -cf felix_sweep.tar felix_sweep_*.tsv fig3_locus_*.tsv
!gzip felix_sweep.tar
!gsutil cp felix_sweep.tar.gz $WORKSPACE_BUCKET/scratch/kvg/
!tar -tzvf felix_sweep.tar.gz

## Caveats

- **Only hits inside each bundle's top-50-per-arm file are screened.** A run reporting more than 50 hits in an arm has the remainder unscreened; the summary counts them so the gap is visible.
- **Retained ≠ real.** It means no detectable SPA fallback, a plausible case-carrier count, and controls that follow Hardy–Weinberg (`excess_het`; main and ru_biallelic arms only, and it can only see an excess at minor-allele frequency of about 0.23 or more). Cross-arm corroboration and recurrence are the next filters, and neither substitutes for replication.
- **λ is untouched by the screen.** The per-phenotype table reports λ as FELIX computed it. A run with λ ≈ 0.6 and zero retained hits is still a run whose repeat arm is mis-calibrated.
- Delete `harvest_cache/` to force a re-harvest after a new batch of runs completes.